In [8]:
!pip install transformers datasets sentencepiece -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 6.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-cu

In [9]:
import pandas as pd
import torch
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from datasets import Dataset


In [10]:
# y_train = pd.read_csv('/content/drive/MyDrive/fraud_detection_pipeline/data/training_&_testing/y_train.csv').squeeze().astype(int)
X_train = pd.read_csv('/content/drive/MyDrive/fraud_detection_pipeline/data/training_&_testing/X_train.csv')
y_train = pd.read_csv('/content/drive/MyDrive/fraud_detection_pipeline/data/training_&_testing/y_train.csv')
X_test = pd.read_csv('/content/drive/MyDrive/fraud_detection_pipeline/data/training_&_testing/X_test.csv')
y_test = pd.read_csv('/content/drive/MyDrive/fraud_detection_pipeline/data/training_&_testing/y_test.csv')
X_train['user_sequence'] = X_train['user_sequence'].fillna('')
X_test['user_sequence'] = X_test['user_sequence'].fillna('')

X_train=X_train['user_sequence'] ;
X_test=X_test['user_sequence'] ;
print (X_train,y_train)
#

0        2023-05-16 10:04:52 edr False False False Fals...
1        2023-01-18 06:48:27 cdr False False False Fals...
2        2023-08-20 15:36:16 edr False False False True...
3        2023-12-22 17:14:10 edr False False False Fals...
4        2023-12-05 02:05:05 ipdr False True False Fals...
                               ...                        
11995    2023-06-02 09:05:45 ipdr False False False Fal...
11996    2023-05-30 15:33:44 ipdr False True False Fals...
11997    2023-06-23 09:56:29 edr False False False True...
11998    2023-02-06 17:09:17 cdr True False False False...
11999    2023-06-29 11:02:26 ipdr False False False Fal...
Name: user_sequence, Length: 12000, dtype: object        is_fraud
0         False
1         False
2         False
3         False
4         False
...         ...
11995     False
11996     False
11997     False
11998      True
11999     False

[12000 rows x 1 columns]


In [5]:
pip install -U transformers


In [14]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [15]:
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import torch

# Step 1: Tokenizer
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

# Step 2: Prepare Datasets
train_dataset = Dataset.from_dict({"text": X_train.tolist(), "label": y_train.squeeze().tolist()})
test_dataset = Dataset.from_dict({"text": X_test.tolist(), "label": y_test.squeeze().tolist()})

def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True)

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)
train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

# Step 3: Load Model
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

# Step 4: TrainingArguments
from transformers import TrainingArguments
#just for fater training ,
train_dataset = train_dataset.select(range(100))# delete when actually need to train
test_dataset = test_dataset.select(range(100))# delete when actually need to train

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir="./logs",
    logging_steps=10
)


# Step 5: Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
   compute_metrics=compute_metrics
)

# Step 6: Train
trainer.train()


metrics = trainer.evaluate()
print(metrics)
# 688b074d285c8217402167e0d970e4dc5d95c21a


Map:   0%|          | 0/12000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss
10,0.457800
20,0.227900
30,0.356600


{'eval_loss': 0.48597007989883423, 'eval_accuracy': 0.85, 'eval_f1': 0.0, 'eval_precision': 0.0, 'eval_recall': 0.0, 'eval_runtime': 88.718, 'eval_samples_per_second': 1.127, 'eval_steps_per_second': 0.147, 'epoch': 3.0}


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [16]:
for metric, value in metrics.items():
    print(f'{metric}: {value}')

eval_loss: 0.48597007989883423
eval_accuracy: 0.85
eval_f1: 0.0
eval_precision: 0.0
eval_recall: 0.0
eval_runtime: 88.718
eval_samples_per_second: 1.127
eval_steps_per_second: 0.147
epoch: 3.0


In [17]:
model.save_pretrained("/content/drive/MyDrive/fraud_detection_pipeline/model")
tokenizer.save_pretrained("/content/drive/MyDrive/fraud_detection_pipeline/model")


('/content/drive/MyDrive/fraud_detection_pipeline/model/tokenizer_config.json',
 '/content/drive/MyDrive/fraud_detection_pipeline/model/special_tokens_map.json',
 '/content/drive/MyDrive/fraud_detection_pipeline/model/vocab.txt',
 '/content/drive/MyDrive/fraud_detection_pipeline/model/added_tokens.json',
 '/content/drive/MyDrive/fraud_detection_pipeline/model/tokenizer.json')

load model that is already trained

In [18]:
model = DistilBertForSequenceClassification.from_pretrained("/content/drive/MyDrive/fraud_detection_pipeline/model")
tokenizer = DistilBertTokenizerFast.from_pretrained("/content/drive/MyDrive/fraud_detection_pipeline/model")



In [ ]:
from transformers import DistilBertTokenizerFast, DistilBertModel
import torch
from tqdm import tqdm

# Load tokenizer and saved fine-tuned model
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")
model = DistilBertModel.from_pretrained("/content/drive/MyDrive/fraud_detection_pipeline/model")  # adjust path
model.eval()  # set model to evaluation mode

# Move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Function to get embeddings
def get_cls_embeddings(text_list):
    embeddings = []
    with torch.no_grad():
        for text in tqdm(text_list):
            inputs = tokenizer(text, return_tensors="pt", truncation=True, padding="max_length", max_length=128)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            outputs = model(**inputs)
            cls_embedding = outputs.last_hidden_state[:, 0, :].squeeze().cpu().numpy()  # [CLS] token
            embeddings.append(cls_embedding)
    return embeddings
# Use This to Get Train/Test Embeddings:
train_embeddings = get_cls_embeddings(X_train.tolist())
test_embeddings = get_cls_embeddings(X_test.tolist())


  0%|          | 40/12000 [00:25<2:05:46,  1.58it/s]


In [ ]:
import numpy as np
import os

# Create a directory for embeddings if not exist
# os.makedirs("embeddings", exist_ok=True)
# /content/drive/MyDrive/fraud_detection_pipeline/data/embeddings
# Save embeddings
np.save("/content/drive/MyDrive/fraud_detection_pipeline/data/embeddings/train_cls_embeddings.npy", np.array(train_embeddings))
np.save("/content/drive/MyDrive/fraud_detection_pipeline/data/embeddings/test_cls_embeddings.npy", np.array(test_embeddings))

 How to Load the Model and Tokenizer Later (No Retraining Needed)

In [ ]:
# from transformers import DistilBertTokenizer, DistilBertForSequenceClassification


# # (Next time, directly load without retraining)
# tokenizer = DistilBertTokenizer.from_pretrained('/content/drive/MyDrive/fraud_detection_pipeline/model/transformer_model')
# model = DistilBertForSequenceClassification.from_pretrained('/content/drive/MyDrive/fraud_detection_pipeline/model/transformer_model')

# # Evaluate

In [ ]:
!jupyter nbconvert --to python "/content/drive/MyDrive/fraud_detection_pipeline/src/transformer.ipynb"


In [ ]:
import shutil
shutil.copy("/content/drive/MyDrive/Colab Notebooks/transformer.ipynb", "/content/drive/MyDrive/fraud_detection_pipeline/src/transformer.ipynb")
# !cp /content/drive/MyDrive/Colab Notebooks/ingest.ipynb /content/drive/MyDrive/fraud_detection_pipeline/src/ingest.ipynb
